In [ ]:
# %% ==============================
# 0. 기본 import & 설정
# %% ==============================
import os
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from gymnasium import spaces
from typing import Optional  # Python 3.9

import matplotlib.pyplot as plt  # 시각화 추가

import myosuite  # MyoSuite:> Registering Myo Envs 나오면 OK


# %% ==============================
# 1. Config
# %% ==============================
class CFG:
    # --- env 설정 ---
    ENV_ID = "motorFingerPoseFixed-v0"   # 안 되면 나중에 "myoFingerPoseFixed-v0" 도 테스트 가능

    # real data frame index window (딱밤 동작이 들어있는 구간)
    WINDOW_START = 800
    WINDOW_END   = 950    # slice [800, 950) → 대략 150 프레임

    EPISODE_LEN  = 150    # 아래에서 WINDOW 길이에 맞춰 다시 세팅할 거라 일단 의미 없음

    # --- reward 설정 ---
    REWARD_SCALE = 0.05   # tracking (||q - target||^2) 계수
    W_SHAPE      = 0.10   # 세 관절 패턴(초반 flex / 후반 extend) 보상 계수
    W_FINAL      = 0.50   # 마지막 프레임에서 fully extend에 대한 보너스 계수

    # --- PPO 설정 ---
    TOTAL_UPDATES     = 3000    # 실험용으로 2000으로 좀 줄여놓음 (필요하면 10000으로 다시 올려)
    STEPS_PER_UPDATE  = 1024    # 한 업데이트 전에 모을 step 수
    GAMMA             = 0.99
    LAMBDA            = 0.95
    CLIP_EPS          = 0.2
    LR                = 1e-5
    BATCH_SIZE        = 256
    PPO_EPOCHS        = 5

    # --- BC-like warm start 설정 ---
    BC_KP            = 1.0       # "expert action ≈ Kp * (target - q)" 용 gain
    BC_LAMBDA        = 0.1       # BC loss 가중치
    BC_WARM_UPDATES  = 500       # 이 업데이트 수까지만 BC loss 적용

    SEED   = 42
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_global_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)


cfg = CFG()
set_global_seed(cfg.SEED)


# %% ==============================
# 2. 중지 관절각 타겟 불러오기 + 800~950 윈도우 잘라 쓰기
# ==============================
def load_middle_finger_targets(npy_path: str, cfg: CFG) -> np.ndarray:
    """
    myohand_joint_angles_23dof_stable_signed.npy 에서
    중지 MCP/PIP/DIP flexion (qpos[11], qpos[13], qpos[14])만 뽑아서 (T_full, 3) 반환,
    그리고 [WINDOW_START:WINDOW_END] 구간만 잘라서 사용.
    """
    qpos_seq = np.load(npy_path)   # (T_full, 23) 예상
    print("[INFO] Loaded NPY:", npy_path, "shape =", qpos_seq.shape)

    # 중지 MCP/PIP/DIP flexion 인덱스
    idx = [11, 13, 14]
    middle_full = qpos_seq[:, idx]   # (T_full, 3)

    print("[INFO] Full middle finger angles shape:", middle_full.shape)
    print("[DEBUG] Full first frame (rad):", middle_full[0])
    print("[DEBUG] Full first frame (deg):", np.rad2deg(middle_full[0]))

    # --- 800~950 구간만 사용 ---
    start = cfg.WINDOW_START
    end   = cfg.WINDOW_END
    middle_win = middle_full[start:end]   # (T_win, 3)

    print(f"[INFO] Using window [{start}:{end}) → shape:", middle_win.shape)
    print("[DEBUG] Window first frame (deg):", np.rad2deg(middle_win[0]))
    print("[DEBUG] Window last  frame (deg):", np.rad2deg(middle_win[-1]))

    return middle_win.astype(np.float32)


# 실제 NPY 경로
NPY_PATH = r"C:\Users\Donggyu\Downloads\myosuite\myohand_joint_angles_23dof_stable_signed.npy"
target_angles = load_middle_finger_targets(NPY_PATH, cfg)

# EPISODE_LEN을 윈도우 길이에 맞추기
cfg.EPISODE_LEN = target_angles.shape[0]
print("[INFO] cfg.EPISODE_LEN set to T_win =", cfg.EPISODE_LEN)


# %% ==============================
# 3. motorFinger wrapper env (패턴 reward + final-state bonus)
# ==============================
class MotorFingerTrajEnv(gym.Env):
    """
    motorFingerPoseFixed-v0 을 싸서,
    - q = [IFmcp, IFpip, IFdip] 현재 각도
    - target_angles[t] = real data (딱밤 구간)
    를 따라가도록 만드는 imitation env.

    reward:
      r = r_track + r_shape + r_final
        - r_track:  - REWARD_SCALE * || q - target ||^2
        - r_shape:  초반(0~0.5)은 flex_ref, 후반(0.5~1)은 ext_ref 에 가까울수록 보상
        - r_final:  마지막 timestep에서만 ext_ref 에 가까울수록 추가 보상
    """

    metadata = {"render_modes": []}

    def __init__(self, target_angles: np.ndarray, cfg: CFG):
        super().__init__()
        self.cfg = cfg
        self.base_env = gym.make(cfg.ENV_ID)
        self.sim = self.base_env.unwrapped.sim

        self.target_angles = target_angles       # (T_win, 3)
        self.T = target_angles.shape[0]          # T_win

        # motorFinger joint 이름: IFmcp, IFpip, IFdip
        mj_model = self.sim.model
        j_mcp = mj_model.joint_name2id("IFmcp")
        j_pip = mj_model.joint_name2id("IFpip")
        j_dip = mj_model.joint_name2id("IFdip")
        j_ids = np.array([j_mcp, j_pip, j_dip], dtype=int)
        self.qpos_adr = mj_model.jnt_qposadr[j_ids]   # (3,)

        # --- 관찰 공간: [q(3), target(3), time_frac] = 7차원 ---
        low = -np.ones(7, dtype=np.float32) * 10.0
        high = np.ones(7, dtype=np.float32) * 10.0
        self.observation_space = spaces.Box(low=low, high=high, dtype=np.float32)

        # --- action space: motorFinger의 action_space 그대로 사용 ---
        self.action_space = self.base_env.action_space

        self.t = 0  # time index

        # ===== 패턴 보상용 flex_ref / ext_ref 계산 (세 관절 모두) =====
        angles = self.target_angles  # (T_win, 3)
        split = int(0.5 * self.T)    # 앞/뒤 반으로 나눔

        # 초반(말린 포즈) 평균
        self.flex_ref = angles[:split].mean(axis=0)   # (3,)
        # 후반(펴진 포즈) 평균
        self.ext_ref = angles[split:].mean(axis=0)    # (3,)

        print(f"[INFO] MotorFingerTrajEnv: obs_dim={self.observation_space.shape[0]}, "
              f"act_dim={self.action_space.shape[0]}, T_win={self.T}")
        print("[INFO] flex_ref (deg):", np.rad2deg(self.flex_ref))
        print("[INFO] ext_ref  (deg):", np.rad2deg(self.ext_ref))

    # ---- internal helpers ----
    def _get_q(self) -> np.ndarray:
        # 현재 IFmcp, IFpip, IFdip 각도 (rad) 3개
        qpos = self.sim.data.qpos
        return qpos[self.qpos_adr].copy()

    def _build_obs(self) -> np.ndarray:
        q = self._get_q()
        idx = min(self.t, self.T - 1)
        target = self.target_angles[idx]
        frac = np.array([self.t / float(self.cfg.EPISODE_LEN)], dtype=np.float32)
        obs = np.concatenate([q.astype(np.float32), target.astype(np.float32), frac], axis=0)
        return obs

    # ---- Gym API ----
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.base_env.reset(seed=seed)
        self.t = 0

        # 초기 포즈를 window 첫 프레임(=800프레임 상태)로 강제 세팅
        qpos = self.sim.data.qpos.copy()
        qpos[self.qpos_adr] = self.target_angles[0]
        self.sim.data.qpos[:] = qpos
        self.sim.forward()

        obs = self._build_obs()
        info = {}
        return obs, info

    def step(self, action):
        # base env step
        _, _, terminated_base, truncated_base, info = self.base_env.step(action)

        # 현재 q / target
        q = self._get_q()                         # (3,)
        idx = min(self.t, self.T - 1)
        target = self.target_angles[idx]          # (3,)
        err = q - target                          # (3,)

        # ---- (1) tracking term ----
        r_track = - self.cfg.REWARD_SCALE * float(np.sum(err ** 2))

        # ---- (2) shape term (세 관절 패턴: 초반 flex, 후반 extend) ----
        phi = self.t / float(self.cfg.EPISODE_LEN)   # [0, 1]
        if phi < 0.5:
            ref = self.flex_ref   # 초반에는 flex_ref (말린 포즈) 근처
        else:
            ref = self.ext_ref    # 후반에는 ext_ref (펴진 포즈) 근처

        shape_err = q - ref
        r_shape = - self.cfg.W_SHAPE * float(np.sum(shape_err ** 2))

        # ---- (3) final-state bonus ----
        if self.t == (self.cfg.EPISODE_LEN - 1):
            final_err = q - self.ext_ref
            r_final = - self.cfg.W_FINAL * float(np.sum(final_err ** 2))
        else:
            r_final = 0.0

        reward = r_track + r_shape + r_final

        # time step 업데이트
        self.t += 1
        terminated = self.t >= self.cfg.EPISODE_LEN
        truncated = False

        obs = self._build_obs()
        done = (terminated or terminated_base or truncated or truncated_base)
        return obs, reward, done, truncated, info

    def render(self):
        return self.base_env.render()

    def close(self):
        self.base_env.close()


# 기본 env (single run용)
env = MotorFingerTrajEnv(target_angles, cfg)


# %% ==============================
# 4. PPO Actor-Critic 정의
# ==============================
class ActorCritic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        hidden = 128

        self.actor = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, action_dim),
        )
        # log_std는 learnable parameter
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        self.critic = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        raise NotImplementedError

    def act(self, state):
        """
        state: (state_dim,) numpy
        return: action (numpy), logprob (float), value (float)
        """
        if not isinstance(state, torch.Tensor):
            state_t = torch.from_numpy(state).float().unsqueeze(0)
        else:
            state_t = state.unsqueeze(0)

        mu = self.actor(state_t)             # (1, A)
        std = torch.exp(self.log_std)        # (A,)
        dist = torch.distributions.Normal(mu, std)
        action = dist.sample()               # (1, A)
        log_prob = dist.log_prob(action).sum(dim=-1)  # (1,)
        value = self.critic(state_t).squeeze(-1)      # (1,)

        return action.detach().cpu().numpy()[0], log_prob.item(), value.item()

    def evaluate_actions(self, states, actions):
        """
        states: (N, state_dim)
        actions: (N, action_dim)
        returns: log_probs (N,), entropy (N,), values (N,)
        """
        mu = self.actor(states)              # (N, A)
        std = torch.exp(self.log_std)        # (A,)
        dist = torch.distributions.Normal(mu, std)

        log_probs = dist.log_prob(actions).sum(dim=-1)  # (N,)
        entropy = dist.entropy().sum(dim=-1)            # (N,)
        values = self.critic(states).squeeze(-1)        # (N,)

        return log_probs, entropy, values


# %% ==============================
# 5. GAE
# ==============================
def compute_gae(rewards, values, dones, gamma, lam):
    """
    rewards, values, dones: (T,)
    """
    T = len(rewards)
    adv = np.zeros(T, dtype=np.float32)
    last_gae = 0.0
    for t in reversed(range(T)):
        next_value = values[t + 1] if t + 1 < T else 0.0
        next_non_terminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * next_value * next_non_terminal - values[t]
        last_gae = delta + gamma * lam * next_non_terminal * last_gae
        adv[t] = last_gae
    returns = adv + values
    return adv, returns


# %% ==============================
# 6. GAE + PPO 학습 루프 (+ BC-like warm start) + 로그 기록
# ==============================
def ppo_train(env: MotorFingerTrajEnv, cfg: CFG, run_name: str = "run") -> tuple[ActorCritic, dict]:
    device = torch.device(cfg.DEVICE)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    ac = ActorCritic(state_dim, action_dim).to(device)
    optimizer = torch.optim.Adam(ac.parameters(), lr=cfg.LR)

    global_step = 0

    # 로그용 리스트
    log_updates = []
    log_avg_return = []
    log_avg_traj_err = []

    for update in range(1, cfg.TOTAL_UPDATES + 1):
        states = []
        actions = []
        rewards = []
        dones = []
        old_log_probs = []
        values = []
        ep_returns = []
        ep_traj_errs = []

        steps_collected = 0

        # ----- rollout 수집 -----
        while steps_collected < cfg.STEPS_PER_UPDATE:
            state, _ = env.reset()
            done = False
            t = 0
            ep_ret = 0.0
            ep_err_sum = 0.0
            ep_len = 0

            while (not done) and (t < cfg.EPISODE_LEN) and (steps_collected < cfg.STEPS_PER_UPDATE):
                action, logp, value = ac.act(state)
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated

                states.append(state)
                actions.append(action)
                rewards.append(reward)
                dones.append(done)
                old_log_probs.append(logp)
                values.append(value)

                ep_ret += reward

                # tracking error 측정 (q vs target)
                q = env._get_q()
                idx = min(env.t, env.T - 1)
                target = env.target_angles[idx]
                ep_err_sum += np.linalg.norm(q - target)
                ep_len += 1

                state = next_state
                t += 1
                steps_collected += 1
                global_step += 1

            if ep_len > 0:
                ep_returns.append(ep_ret)
                ep_traj_errs.append(ep_err_sum / ep_len)

        # numpy 변환
        states = np.array(states, dtype=np.float32)
        actions = np.array(actions, dtype=np.float32)
        rewards = np.array(rewards, dtype=np.float32)
        dones = np.array(dones, dtype=np.bool_)
        values = np.array(values, dtype=np.float32)

        # ----- GAE -----
        advantages, returns = compute_gae(rewards, values, dones,
                                          gamma=cfg.GAMMA, lam=cfg.LAMBDA)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        states_t = torch.from_numpy(states).float().to(device)
        actions_t = torch.from_numpy(actions).float().to(device)
        old_log_probs_t = torch.from_numpy(np.array(old_log_probs, dtype=np.float32)).to(device)
        returns_t = torch.from_numpy(returns).float().to(device)
        adv_t = torch.from_numpy(advantages).float().to(device)

        # ----- PPO 업데이트 -----
        dataset_size = states_t.shape[0]
        idxs = np.arange(dataset_size)
        last_loss = 0.0

        for epoch in range(cfg.PPO_EPOCHS):
            np.random.shuffle(idxs)
            for start in range(0, dataset_size, cfg.BATCH_SIZE):
                end = start + cfg.BATCH_SIZE
                batch_idx = idxs[start:end]

                b_states = states_t[batch_idx]
                b_actions = actions_t[batch_idx]
                b_old_logp = old_log_probs_t[batch_idx]
                b_returns = returns_t[batch_idx]
                b_adv = adv_t[batch_idx]

                new_logp, entropy, values_pred = ac.evaluate_actions(b_states, b_actions)
                ratio = torch.exp(new_logp - b_old_logp)

                surr1 = ratio * b_adv
                surr2 = torch.clamp(ratio,
                                    1.0 - cfg.CLIP_EPS,
                                    1.0 + cfg.CLIP_EPS) * b_adv

                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = nn.MSELoss()(values_pred, b_returns)
                entropy_loss = -entropy.mean()

                # ----- BC-like warm start loss -----
                bc_loss = torch.tensor(0.0, device=device)
                if update <= cfg.BC_WARM_UPDATES and cfg.BC_LAMBDA > 0.0:
                    q_t   = b_states[:, 0:3]
                    tgt_t = b_states[:, 3:6]
                    err_t = tgt_t - q_t       # (N, 3)

                    expert_act = torch.zeros_like(ac.actor(b_states))  # (N, action_dim)
                    expert_act[:, 0:3] = cfg.BC_KP * err_t             # 앞 3개만 사용 (heuristic)

                    mu_pred = ac.actor(b_states)
                    bc_loss = ((mu_pred - expert_act) ** 2).mean() * cfg.BC_LAMBDA

                loss = actor_loss + 0.5 * critic_loss + 0.001 * entropy_loss + bc_loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                last_loss = loss.item()

        avg_return = float(np.mean(ep_returns)) if len(ep_returns) > 0 else 0.0
        avg_err = float(np.mean(ep_traj_errs)) if len(ep_traj_errs) > 0 else 0.0

        log_updates.append(update)
        log_avg_return.append(avg_return)
        log_avg_traj_err.append(avg_err)

        if (update % 10) == 0 or update == 1:
            print(f"[{run_name}] [Update {update:4d}/{cfg.TOTAL_UPDATES}] "
                  f"Steps: {global_step:7d}  AvgReturn: {avg_return:7.3f}  "
                  f"MeanTrackErr: {avg_err:7.4f}  LastLoss: {last_loss:7.4f}")

    print(f"=== Training finished: {run_name} ===")

    # policy 저장
    save_path = f"ppo_motorfinger_actor_{run_name}.pth"
    actor_state = {
        "actor": ac.actor.state_dict(),
        "log_std": ac.log_std.detach().cpu(),
    }
    torch.save(actor_state, save_path)
    print(f"[INFO] Saved actor (policy) weights to {save_path}")

    logs = {
        "update": np.array(log_updates),
        "avg_return": np.array(log_avg_return),
        "avg_traj_err": np.array(log_avg_traj_err),
    }

    return ac, logs


# %% ==============================
# 7. 평가: trained vs random 비교
# ==============================
def eval_policy(env: MotorFingerTrajEnv,
                policy: Optional[ActorCritic],
                cfg: CFG,
                episodes: int = 5):
    def run_episode(use_policy: bool):
        state, _ = env.reset()
        done = False
        ep_ret = 0.0
        traj_err = []

        t = 0
        while not done and t < cfg.EPISODE_LEN:
            if use_policy and policy is not None:
                action, _, _ = policy.act(state)
            else:
                low = env.action_space.low
                high = env.action_space.high
                action = np.random.uniform(low, high)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_ret += reward

            q = env._get_q()
            idx = min(env.t, env.T - 1)
            target = env.target_angles[idx]
            traj_err.append(np.linalg.norm(q - target))

            state = next_state
            t += 1

        return ep_ret, float(np.mean(traj_err))

    print("===== Eval random policy =====")
    rets, errs = [], []
    for _ in range(episodes):
        r, e = run_episode(use_policy=False)
        rets.append(r)
        errs.append(e)
    print(f"[Random]   mean return={np.mean(rets):.3f}, mean tracking error={np.mean(errs):.4f}")

    if policy is not None:
        print("===== Eval trained policy =====")
        rets, errs = [], []
        for _ in range(episodes):
            r, e = run_episode(use_policy=True)
            rets.append(r)
            errs.append(e)
        print(f"[Trained]  mean return={np.mean(rets):.3f}, mean tracking error={np.mean(errs):.4f}")


# %% ==============================
# 8. multi-seed + hyperparameter 실험 + 시각화
# ==============================
def run_experiments():
    """
    - 여러 seed, 여러 하이퍼파라미터 설정으로 학습
    - 각 설정별로 업데이트 vs (episode return / tracking error) 곡선
    - 마지막 업데이트 기준 tracking error bar plot
    """
    # 실험할 세팅들 (baseline + 변형 두 개 예시)
    experiment_configs = [
        {
            "name": "baseline",
            "cfg_mod": dict(
                LR=1e-5,
                REWARD_SCALE=0.05,
                W_SHAPE=0.10,
                W_FINAL=0.50,
                BC_LAMBDA=0.1,
                TOTAL_UPDATES=2000,
            ),
        },
        {
            "name": "strong_shape",
            "cfg_mod": dict(
                LR=1e-5,
                REWARD_SCALE=0.05,
                W_SHAPE=0.20,   # shape term 가중치 ↑
                W_FINAL=0.50,
                BC_LAMBDA=0.1,
                TOTAL_UPDATES=2000,
            ),
        },
        {
            "name": "higher_lr",
            "cfg_mod": dict(
                LR=3e-5,        # learning rate ↑
                REWARD_SCALE=0.05,
                W_SHAPE=0.10,
                W_FINAL=0.50,
                BC_LAMBDA=0.1,
                TOTAL_UPDATES=2000,
            ),
        },
    ]

    seeds = [42, 123, 777]  # 여러 seed

    all_results = {}  # exp_name -> list of logs

    for exp in experiment_configs:
        exp_name = exp["name"]
        cfg_mod = exp["cfg_mod"]
        all_results[exp_name] = []

        print(f"\n========== Experiment: {exp_name} ==========")

        for seed in seeds:
            print(f"\n--- Seed {seed} ---")
            # 새로운 cfg 객체 생성 후 수정
            exp_cfg = CFG()
            for k, v in cfg_mod.items():
                setattr(exp_cfg, k, v)
            exp_cfg.SEED = seed

            set_global_seed(seed)

            # 타겟 / env 새로 생성
            target_angles_local = load_middle_finger_targets(NPY_PATH, exp_cfg)
            exp_cfg.EPISODE_LEN = target_angles_local.shape[0]
            env_local = MotorFingerTrajEnv(target_angles_local, exp_cfg)

            run_name = f"{exp_name}_seed{seed}"
            _, logs = ppo_train(env_local, exp_cfg, run_name=run_name)
            env_local.close()

            logs["seed"] = seed
            all_results[exp_name].append(logs)

    # 시각화
    plot_results(all_results, seeds)


def plot_results(all_results: dict, seeds: list[int]):
    """
    all_results[exp_name] = [ { "update":..., "avg_return":..., "avg_traj_err":..., "seed":... }, ... ]
    """
    # (1) Episode Return learning curve (mean ± std over seeds)
    plt.figure(figsize=(10, 5))
    for exp_name, logs_list in all_results.items():
        min_len = min(len(l["update"]) for l in logs_list)
        updates = logs_list[0]["update"][:min_len]
        returns_mat = np.stack([l["avg_return"][:min_len] for l in logs_list], axis=0)
        mean_ret = returns_mat.mean(axis=0)
        std_ret = returns_mat.std(axis=0)

        plt.plot(updates, mean_ret, label=exp_name)
        plt.fill_between(updates, mean_ret - std_ret, mean_ret + std_ret, alpha=0.2)

    plt.xlabel("PPO Update")
    plt.ylabel("Episode Return")
    plt.title("PPO Learning Curves (mean ± std over seeds)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("ppo_learning_curves_return.png", dpi=200)
    plt.close()

    # (2) Tracking Error learning curve (mean ± std over seeds)
    plt.figure(figsize=(10, 5))
    for exp_name, logs_list in all_results.items():
        min_len = min(len(l["update"]) for l in logs_list)
        updates = logs_list[0]["update"][:min_len]
        err_mat = np.stack([l["avg_traj_err"][:min_len] for l in logs_list], axis=0)
        mean_err = err_mat.mean(axis=0)
        std_err = err_mat.std(axis=0)

        plt.plot(updates, mean_err, label=exp_name)
        plt.fill_between(updates, mean_err - std_err, mean_err + std_err, alpha=0.2)

    plt.xlabel("PPO Update")
    plt.ylabel("Mean Tracking Error (rad)")
    plt.title("Tracking Error vs PPO Update (mean ± std over seeds)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("ppo_learning_curves_trackerr.png", dpi=200)
    plt.close()

    # (3) 마지막 업데이트 기준 tracking error bar plot
    plt.figure(figsize=(8, 5))
    names = []
    means = []
    stds = []
    for exp_name, logs_list in all_results.items():
        final_errs = [l["avg_traj_err"][-1] for l in logs_list]
        names.append(exp_name)
        means.append(np.mean(final_errs))
        stds.append(np.std(final_errs))

    x = np.arange(len(names))
    plt.bar(x, means, yerr=stds, capsize=5)
    plt.xticks(x, names, rotation=20)
    plt.ylabel("Final mean tracking error (rad)")
    plt.title("Final Tracking Error (mean ± std over seeds)")
    plt.tight_layout()
    plt.savefig("ppo_final_tracking_error_bar.png", dpi=200)
    plt.close()

    print("[INFO] Saved plots:")
    print(" - ppo_learning_curves_return.png")
    print(" - ppo_learning_curves_trackerr.png")
    print(" - ppo_final_tracking_error_bar.png")


# %% ==============================
# 9. main: single run + multi-experiment 둘 다 옵션
# ==============================
if __name__ == "__main__":
    # 1) 일단 baseline single run + eval (원래 코드)
    print("\n===== Single PPO Training (motorFingerPoseFixed, window 800~950 + BC-warm + init-from-800) =====")
    policy, single_logs = ppo_train(env, cfg, run_name="single_baseline")

    print("\n===== Policy Evaluation (single baseline) =====")
    eval_policy(env, policy, cfg, episodes=5)

    # 2) multi-seed + hyperparameter 실험 + 시각화
    print("\n===== Multi-seed / Hyperparam Experiments =====")
    run_experiments()

    env.close()
